In [1]:
%pip install nltk

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [28]:
import nltk
import numpy as np
import random


import bs4 as bs
import urllib.request
import re

nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\niaco\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\niaco\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

1. Load in Data

<small> I decided to use the Project Gutenberg txt file with the complete works of shakespeare
</small>

In [ ]:
#Access shaekespeare project gutenberg file
url = 'https://www.gutenberg.org/cache/epub/100/pg100.txt'
headers = {'User-Agent': 'Mozilla/5.0'}

try:
    request = urllib.request.Request(url, headers=headers)
    with urllib.request.urlopen(request) as response:
        raw_text = response.read().decode('utf-8').lower()

    # Cleaning: Keep letters, spaces, periods, and apostrophes
    article_text = re.sub(r'[^a-z.\' ]', '', raw_text)
    
    # Slicing to skip the legal header
    start_idx = article_text.find("the complete works of william shakespeare")
    if start_idx != -1:
        article_text = article_text[start_idx:]

    print(f"Dataset ready. Total characters: {len(article_text)}")
except Exception as e:
    print(f"Error: {e}")

Dataset ready. Total characters: 4996028


 # Simple N-Gram Model:

2. Re-Train Original Model from Class

In [ ]:
# Configuration
n = 3  # Using 3-word sequences (Trigrams)
ngrams = {}
words_tokens = nltk.word_tokenize(article_text)
ngrams_original = {}

for i in range(len(words_tokens) - n):
    seq = ' '.join(words_tokens[i:i+n])
    next_word = words_tokens[i+n]

    if seq not in ngrams_original:
        ngrams_original[seq] = [] 
    
    ngrams_original[seq].append(next_word)

print("Original model trained.")

Original model trained.


3. Update N-gram Model by inluding Probabilities

In [ ]:
# Training: Building the Probability/Frequency Dictionary
for i in range(len(words_tokens) - n):
    seq = ' '.join(words_tokens[i:i+n])
    next_word = words_tokens[i+n]

    if seq not in ngrams:
        ngrams[seq] = {} 
    
    # Count occurrences of the next word following this sequence
    if next_word not in ngrams[seq]:
        ngrams[seq][next_word] = 1
    else:
        ngrams[seq][next_word] += 1

print(f"Model trained with {len(ngrams)} unique n-gram sequences.")

Model trained with 744759 unique n-gram sequences.


4. Comparison of Original and Count Probability N-gram Models

In [ ]:
start_index = 9997
# 1. ORIGINAL STYLE (Random from list)
search_original = ' '.join(words_tokens[start_index:start_index+n])
output_orig = search_original
for _ in range(30):
    if search_original in ngrams_original:
        next_w = random.choice(ngrams_original[search_original])
        output_orig += ' ' + next_w
        search_original = ' '.join(nltk.word_tokenize(output_orig)[-n:])

# 3. MAX PROBABILITY (Pick the absolute #1 word)
search_max = ' '.join(words_tokens[start_index:start_index+n])
output_max = search_max
for _ in range(30):
    if search_max in ngrams:
        next_w = max(ngrams[search_max], key=ngrams[search_max].get)
        output_max += ' ' + next_w
        search_max = ' '.join(nltk.word_tokenize(output_max)[-n:])

print(f"--- COMPARISON ---")
print(f"Original: {output_orig}\n")
print(f"Max Prob: {output_max}")

--- COMPARISON ---
Original: dwellwhateer thy thoughts or thy hearts workings bethy looks should nothing thence but sweetness tell . how like a maid she blushes here.o what authority and show of truthcan cunning sin cover itself

Max Prob: dwellwhateer thy thoughts or thy hearts workings bethy looks should nothing thence but sweetness tell . how like a winter hath my absence beenfrom thee the pleasure of the fleeting yearwhat freezings have


#### Ouput Analysis
<small>
The original n-gram model picks any following word with equal probability. The moment it hit the common sequence "how like a," it had a dozen different directions to go. It randomly rolled the dice and landed on "maid," completely losing the context of the poem it was currently writing. As a result, it moved from Sonnet 93 into a line from the play Much Ado About Nothing.

<p> However, the max probability model stuck to the most statistically significant path which was the opening of Sonnet 97. It looked at every word that follows "how like a" and saw that "winter" appeared more frequently or consistently across the dataset than "maid" or "apple." It chose the "strongest" link. Due to the nature of this model, there is unfortunately little room for creativity in text generation and more often than not it would just produce a replica of the training data.

</small>